# PennyLane Pendigits Smoke

This notebook is an implementation feasibility check for the experimental PennyLane backend. Smoke-subset results are not paper-facing performance evidence and should not be cited as final Pendigits accuracy, trainability, or scaling results.

In [1]:
import os

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from jax import config as jax_config

jax_config.update("jax_enable_x64", True)


In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "src" / "ham_embed_spectral").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing src/ham_embed_spectral")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [3]:
import jax
import jax.numpy as jnp
import optax
import pandas as pd

from ham_embed_spectral.config import ReuploadingModelConfig
from ham_embed_spectral.data.pendigits import PendigitsDataset, prepare_pendigits
from ham_embed_spectral.experiments.train_loop import (
    create_train_state,
    epoch_minibatches,
    evaluate,
    limit_split,
    make_predict_step,
    make_train_step,
)
from ham_embed_spectral.models.pennylane_reuploading import (
    make_pennylane_predict_step,
    make_pennylane_train_step,
    pennylane_forward_state,
    pennylane_probabilities,
)
from ham_embed_spectral.models.reuploading import (
    forward_state,
    init_reuploading_params,
    probabilities,
)
from ham_embed_spectral.quantum.encoders import (
    BlockHamiltonianEncoder,
    FixedRyEncoder,
    FixedRyRzEncoder,
    NonOverlapPatchBlockHamiltonianEncoder,
    PatchSU4Encoder,
    SymmetricHamiltonianEncoder,
    TrainableFrequencyRyEncoder,
    TrainablePatchSU4Encoder,
)
from ham_embed_spectral.quantum.readout import ceil_log2

E0603 16:00:04.571801 3509996 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0603 16:00:04.581794 3509132 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


In [4]:
representation = "sta4"
depth = 4
seed = 0
steps = 50
batch_size = 16
learning_rate = 1e-2
max_train = 128
max_validation = 64
max_test = 128
data_root = ROOT / "data/raw/pendigits"
output_csv = ROOT / "results/tables/pennylane_pendigits_smoke.csv"

encoder_factories = {
    "fixed-ry": FixedRyEncoder,
    "fixed-ry-rz": FixedRyRzEncoder,
    "trainable-frequency-ry": TrainableFrequencyRyEncoder,
    "patch-su4": PatchSU4Encoder,
    "trainable-patch-su4": TrainablePatchSU4Encoder,
    "symmetric-hamiltonian": SymmetricHamiltonianEncoder,
    "block-hamiltonian": BlockHamiltonianEncoder,
    "non-overlap-patch-block-hamiltonian": NonOverlapPatchBlockHamiltonianEncoder,
}
encoder_names = list(encoder_factories)

In [5]:
def limited_dataset(dataset: PendigitsDataset) -> PendigitsDataset:
    return PendigitsDataset(
        train=limit_split(dataset.train, max_train),
        validation=limit_split(dataset.validation, max_validation),
        test=limit_split(dataset.test, max_test),
        representation=dataset.representation,
        input_shape=dataset.input_shape,
        class_values=dataset.class_values,
        feature_mean=dataset.feature_mean,
        feature_std=dataset.feature_std,
    )


def model_config_for(encoder, dataset: PendigitsDataset) -> ReuploadingModelConfig:
    natural_n_qubits = encoder.n_qubits(dataset.input_shape)
    label_n_qubits = ceil_log2(dataset.n_classes)
    n_qubits = natural_n_qubits
    if isinstance(encoder, (SymmetricHamiltonianEncoder, BlockHamiltonianEncoder)):
        n_qubits = max(natural_n_qubits, label_n_qubits)
    elif label_n_qubits > natural_n_qubits:
        raise ValueError(
            f"{type(encoder).__name__} has {natural_n_qubits} qubits, "
            f"but {dataset.n_classes} classes need {label_n_qubits} label qubits"
        )
    return ReuploadingModelConfig(
        input_shape=dataset.input_shape,
        n_classes=dataset.n_classes,
        reupload_depth=depth,
        n_qubits=n_qubits,
        projector_renormalize=True,
    )


def diagnostic_discrepancy(params, encoder, config, batch_x):
    jax_states = jax.vmap(lambda x: forward_state(params, encoder, x, config))(batch_x)
    pennylane_states = jnp.stack(
        [pennylane_forward_state(params, encoder, x, config) for x in batch_x]
    )
    jax_probs = probabilities(params, encoder, batch_x, config)
    pennylane_probs = pennylane_probabilities(params, encoder, batch_x, config)
    return {
        "initial_state_max_abs_diff": float(jnp.max(jnp.abs(jax_states - pennylane_states))),
        "initial_probability_max_abs_diff": float(jnp.max(jnp.abs(jax_probs - pennylane_probs))),
    }


def train_backend(name, params0, encoder, config, dataset):
    optimizer = optax.adam(learning_rate)
    state = create_train_state(params0, optimizer)
    if name == "jax":
        train_step = make_train_step(encoder, config, optimizer)
        predict_step = make_predict_step(encoder, config)
    elif name == "pennylane":
        train_step = make_pennylane_train_step(encoder, config, optimizer)
        predict_step = make_pennylane_predict_step(encoder, config)
    else:
        raise ValueError(name)

    batches = epoch_minibatches(dataset.train, batch_size, seed=seed, shuffle=True)
    for _ in range(steps):
        batch_x, batch_y = next(batches)
        state, _ = train_step(state, batch_x, batch_y)

    return state, {
        "train": evaluate(
            state.params,
            dataset.train,
            predict_step=predict_step,
            batch_size=batch_size,
        ),
        "validation": evaluate(
            state.params,
            dataset.validation,
            predict_step=predict_step,
            batch_size=batch_size,
        ),
        "test": evaluate(
            state.params,
            dataset.test,
            predict_step=predict_step,
            batch_size=batch_size,
        ),
    }

In [6]:
dataset = limited_dataset(
    prepare_pendigits(
        data_root,
        representation=representation,
        validation_fraction=0.1,
        seed=seed,
        standardize=True,
        download=False,
        dtype=jnp.float64,
    )
)
(
    dataset.input_shape,
    dataset.n_classes,
    dataset.train.x.shape,
    dataset.validation.x.shape,
    dataset.test.x.shape,
)

((4, 4), 10, (128, 4, 4), (64, 4, 4), (128, 4, 4))

In [7]:
rows = []
diag_batch = dataset.validation.x[: min(4, int(dataset.validation.y.shape[0]))]

for encoder_name in encoder_names:
    encoder = encoder_factories[encoder_name]()
    config = model_config_for(encoder, dataset)
    params0 = init_reuploading_params(jax.random.PRNGKey(seed), encoder, config)
    discrepancies = diagnostic_discrepancy(params0, encoder, config, diag_batch)

    _, jax_metrics = train_backend("jax", params0, encoder, config, dataset)
    _, pennylane_metrics = train_backend("pennylane", params0, encoder, config, dataset)

    row = {
        "representation": representation,
        "encoder": encoder_name,
        "depth": depth,
        "seed": seed,
        "steps": steps,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "n_qubits": config.n_qubits,
        **discrepancies,
    }
    for backend_name, metrics in (("jax", jax_metrics), ("pennylane", pennylane_metrics)):
        for split_name, split_metrics in metrics.items():
            row[f"{backend_name}_{split_name}_loss"] = split_metrics["loss"]
            row[f"{backend_name}_{split_name}_accuracy"] = split_metrics["accuracy"]
    rows.append(row)
    display(pd.DataFrame([row]))

results = pd.DataFrame(rows)
output_csv.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(output_csv, index=False)
output_csv, results

,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,fixed-ry,4,0,50,16,0.01,16,2.331468e-15,6.661338e-16,...,2.360713,0.25,2.187286,0.226562,1.684678,0.390625,2.360713,0.25,2.187286,0.226562


,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,fixed-ry-rz,4,0,50,16,0.01,16,6.300653e-16,9.436896e-16,...,2.298953,0.203125,2.198006,0.25,1.522701,0.492188,2.298953,0.203125,2.198006,0.25


,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,trainable-frequency-ry,4,0,50,16,0.01,16,2.278663e-15,6.106227e-16,...,2.243872,0.28125,2.366052,0.203125,1.670127,0.460938,2.243872,0.28125,2.366052,0.203125


,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,patch-su4,4,0,50,16,0.01,8,3.273092e-13,5.047907e-13,...,2.475442,0.125,2.410625,0.179688,1.495664,0.5,2.475442,0.125,2.410625,0.179688


,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,trainable-patch-su4,4,0,50,16,0.01,8,2.600041e-13,3.452238e-13,...,2.528872,0.21875,2.318282,0.195312,0.967275,0.84375,2.528872,0.21875,2.318282,0.195312


,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,symmetric-hamiltonian,4,0,50,16,0.01,4,1.913525e-15,1.026956e-15,...,2.003297,0.359375,1.912348,0.398438,1.520337,0.5625,2.003297,0.359375,1.912348,0.398438


,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,block-hamiltonian,4,0,50,16,0.01,4,3.997574e-15,2.248202e-15,...,1.880938,0.390625,1.76649,0.398438,1.449477,0.578125,1.880938,0.390625,1.76649,0.398438


,representation,encoder,depth,seed,steps,batch_size,learning_rate,n_qubits,initial_state_max_abs_diff,initial_probability_max_abs_diff,...,jax_validation_loss,jax_validation_accuracy,jax_test_loss,jax_test_accuracy,pennylane_train_loss,pennylane_train_accuracy,pennylane_validation_loss,pennylane_validation_accuracy,pennylane_test_loss,pennylane_test_accuracy
0,sta4,non-overlap-patch-block-hamiltonian,4,0,50,16,0.01,8,1.117140e-15,1.221245e-15,...,1.75887,0.390625,1.915477,0.3125,1.434356,0.484375,1.75887,0.390625,1.915477,0.3125


(PosixPath('results/tables/pennylane_pendigits_smoke.csv'),
   representation                              encoder  depth  seed  steps  \
 0           sta4                             fixed-ry      4     0     50   
 1           sta4                          fixed-ry-rz      4     0     50   
 2           sta4               trainable-frequency-ry      4     0     50   
 3           sta4                            patch-su4      4     0     50   
 4           sta4                  trainable-patch-su4      4     0     50   
 5           sta4                symmetric-hamiltonian      4     0     50   
 6           sta4                    block-hamiltonian      4     0     50   
 7           sta4  non-overlap-patch-block-hamiltonian      4     0     50   
 
    batch_size  learning_rate  n_qubits  initial_state_max_abs_diff  \
 0          16           0.01        16                2.331468e-15   
 1          16           0.01        16                6.300653e-16   
 2          16         

Optional DYN repeat: set `representation = "dyn"` in the defaults cell and rerun the notebook. The same output CSV path is used, so change `output_csv` first if both STA4 and DYN smoke tables should be kept side by side.